In [0]:
%sql
-- Create 'trend_market_project' folder
CREATE VOLUME IF NOT EXISTS msbabigdata.spark.trend_market_project;

PART 1: Initialize Paths and Ingest Data

In [0]:
# Step 1: Define paths
trips_path = "/Volumes/msbabigdata/spark/trend_market_project/yellow_tripdata_2023_sample.parquet"
zone_path = "/Volumes/msbabigdata/spark/trend_market_project/Taxi_zone_lookup.csv"

In [0]:
# Step 2: Ingest data
df_raw = spark.read.parquet(trips_path)
zone_df = spark.read.option("header", "true").option("inferSchema", "true").csv(zone_path)

In [0]:
# Step 3: Standardize column names to lowercase immediately (fixes the 'Airport_fee' issue)
df_trips = df_raw.toDF(*[c.lower() for c in df_raw.columns])

In [0]:
# --- VALIDATION OF PART 1: Profile the raw data ---
print(f"Raw record count: {df_trips.count()}")
df_trips.select("tpep_pickup_datetime", "fare_amount", "trip_distance").summary().show()

Raw record count: 3024
+-------+------------------+------------------+
|summary|       fare_amount|     trip_distance|
+-------+------------------+------------------+
|  count|              3024|              3024|
|   mean|19.946779100529096|3.8835582010581953|
| stddev| 19.47536322515487| 4.855978738798475|
|    min|            -100.0|               0.0|
|    25%|               8.6|               1.1|
|    50%|              13.5|              1.95|
|    75%|              22.6|              4.02|
|    max|             200.4|             38.02|
+-------+------------------+------------------+



PART 2: Build the Cleaning Pipeline

In [0]:
from pyspark.sql import functions as F

In [0]:
def clean_taxi_data(df):
    return (df
        # 1. Handle Erroneous Dates: Filter strictly for 2023
        .filter(F.year("tpep_pickup_datetime") == 2023)
        
        # 2. Handle Null/Invalid Zone IDs (NYC zones are 1-263)
        .filter(F.col("pulocationid").isNotNull() & (F.col("pulocationid").between(1, 263)))
        .filter(F.col("dolocationid").isNotNull() & (F.col("dolocationid").between(1, 263)))
        
        # 3. Handle Outliers: Fare >= $2.50 (NYC min) and sensible distance
        .filter((F.col("fare_amount") >= 2.5) & (F.col("fare_amount") < 500))
        .filter((F.col("trip_distance") > 0) & (F.col("trip_distance") < 100))
        
        # 4. Remove Duplicate records
        .dropDuplicates()
        
        # 5. Timestamp formatting & Timezone Normalization
        # We explicitly convert the UTC data to New York time
        .withColumn("tpep_pickup_datetime", F.from_utc_timestamp(F.col("tpep_pickup_datetime"), "America/New_York"))
        .withColumn("tpep_dropoff_datetime", F.from_utc_timestamp(F.col("tpep_dropoff_datetime"), "America/New_York"))
        
        # Add a date column for partitioning later
        .withColumn("pickup_date", F.to_date("tpep_pickup_datetime"))
    )

In [0]:
df_cleaned = clean_taxi_data(df_trips)

In [0]:
# --- VALIDATION of PART 2: Check Cleaning Results ---
print(f"Cleaned record count: {df_cleaned.count()}")

Cleaned record count: 2879


In [0]:
# Check for remaining nulls in critical columns
df_cleaned.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in ["pulocationid", "fare_amount"]]).show()

+------------+-----------+
|pulocationid|fare_amount|
+------------+-----------+
|           0|          0|
+------------+-----------+



In [0]:
# Verify Date Range (Should only be 2023)
df_cleaned.select(F.min("tpep_pickup_datetime"), F.max("tpep_pickup_datetime")).show()

+-------------------------+-------------------------+
|min(tpep_pickup_datetime)|max(tpep_pickup_datetime)|
+-------------------------+-------------------------+
|      2022-12-31 19:35:55|      2023-03-31 19:20:37|
+-------------------------+-------------------------+



PART 3: Join with Zone Lookup

In [0]:
# Join cleaned trips with the lookup table
# Use left join to ensure we keep trip records even if a zone ID is weird

# Join 1: Attach names for the Pickup Location
final_df = df_cleaned.join(
    zone_df.select(F.col("LocationID").alias("puloc_id"), 
                   F.col("Borough").alias("pickup_borough"), 
                   F.col("Zone").alias("pickup_zone")), 
    df_cleaned.pulocationid == F.col("puloc_id"), 
    "left"
).drop("puloc_id")

In [0]:
# Join 2: Attach names for the Dropoff Location (Requirement B)
final_df = final_df.join(
    zone_df.select(F.col("LocationID").alias("doloc_id"), 
                   F.col("Borough").alias("dropoff_borough"), 
                   F.col("Zone").alias("dropoff_zone")), 
    final_df.dolocationid == F.col("doloc_id"), 
    "left"
).drop("doloc_id")

In [0]:
# --- VALIDATION of PART 3: Check Join ---
# Ensure we have borough names instead of just IDs
final_df.select("pulocationid", "pickup_borough", "pickup_zone").distinct().limit(5).show()

+------------+--------------+-------------------+
|pulocationid|pickup_borough|        pickup_zone|
+------------+--------------+-------------------+
|          79|     Manhattan|       East Village|
|         140|     Manhattan|    Lenox Hill East|
|         144|     Manhattan|Little Italy/NoLiTa|
|         260|        Queens|           Woodside|
|          75|     Manhattan|  East Harlem South|
+------------+--------------+-------------------+



In [0]:
# --- VALIDATION of PART 3: check both ends of the trip
final_df.select(
    "pulocationid", "pickup_borough", 
    "dolocationid", "dropoff_borough"
).distinct().limit(5).show()

+------------+--------------+------------+---------------+
|pulocationid|pickup_borough|dolocationid|dropoff_borough|
+------------+--------------+------------+---------------+
|          68|     Manhattan|          87|      Manhattan|
|         132|        Queens|         229|      Manhattan|
|         231|     Manhattan|          25|       Brooklyn|
|          68|     Manhattan|         238|      Manhattan|
|          50|     Manhattan|          48|      Manhattan|
+------------+--------------+------------+---------------+



PART 4: Write to Delta Lake

In [0]:
target_table = "msbabigdata.spark.trend_market_cleaned_trips"

(final_df.write
 .format("delta")
 .mode("overwrite")
 .option("overwriteSchema", "true")
 .partitionBy("pickup_date", "pulocationid")
 .saveAsTable(target_table))

In [0]:
# --- VALIDATION of PART 4: Query the Delta Table ---
print("Final Table Sample:")
display(spark.sql(f"SELECT * FROM {target_table} LIMIT 10"))

Final Table Sample:


vendorid,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,ratecodeid,store_and_fwd_flag,pulocationid,dolocationid,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,airport_fee,pickup_date,pickup_borough,pickup_zone,dropoff_borough,dropoff_zone
1,2022-12-31T22:53:13.000Z,2022-12-31T22:59:21.000Z,1.0,1.5,1.0,N,142,239,1,9.3,3.5,0.5,2.85,0.0,1.0,17.15,2.5,0.0,2022-12-31,Manhattan,Lincoln Square East,Manhattan,Upper West Side South
1,2022-12-31T20:30:04.000Z,2022-12-31T20:49:34.000Z,3.0,3.7,1.0,N,142,79,1,21.2,3.5,0.5,5.2,0.0,1.0,31.4,2.5,0.0,2022-12-31,Manhattan,Lincoln Square East,Manhattan,East Village
1,2022-12-31T23:56:28.000Z,2023-01-01T00:09:02.000Z,1.0,5.7,1.0,N,163,116,1,24.7,3.5,0.5,3.0,0.0,1.0,32.7,2.5,0.0,2022-12-31,Manhattan,Midtown North,Manhattan,Hamilton Heights
2,2022-12-31T22:03:17.000Z,2022-12-31T22:16:55.000Z,1.0,3.2,1.0,N,163,41,1,17.0,1.0,0.5,4.4,0.0,1.0,26.4,2.5,0.0,2022-12-31,Manhattan,Midtown North,Manhattan,Central Harlem
2,2022-12-31T23:18:15.000Z,2022-12-31T23:28:06.000Z,1.0,3.3,1.0,N,262,168,1,14.9,1.0,0.5,5.97,0.0,1.0,25.87,2.5,0.0,2022-12-31,Manhattan,Yorkville East,Bronx,Mott Haven/Port Morris
2,2022-12-31T21:17:41.000Z,2022-12-31T21:31:52.000Z,4.0,5.05,1.0,N,4,263,1,23.3,1.0,0.5,5.66,0.0,1.0,33.96,2.5,0.0,2022-12-31,Manhattan,Alphabet City,Manhattan,Yorkville West
2,2022-12-31T19:35:55.000Z,2022-12-31T19:43:59.000Z,4.0,2.11,1.0,N,162,263,1,11.4,1.0,0.5,3.28,0.0,1.0,19.68,2.5,0.0,2022-12-31,Manhattan,Midtown East,Manhattan,Yorkville West
2,2022-12-31T22:21:56.000Z,2022-12-31T22:26:28.000Z,2.0,0.71,1.0,N,164,170,2,6.5,1.0,0.5,0.0,0.0,1.0,11.5,2.5,0.0,2022-12-31,Manhattan,Midtown South,Manhattan,Murray Hill
2,2022-12-31T21:18:42.000Z,2022-12-31T21:39:31.000Z,2.0,5.28,1.0,N,234,151,1,26.8,1.0,0.5,6.36,0.0,1.0,38.16,2.5,0.0,2022-12-31,Manhattan,Union Sq,Manhattan,Manhattan Valley
2,2022-12-31T23:15:33.000Z,2022-12-31T23:32:44.000Z,null,2.99,null,null,246,79,0,18.69,0.0,0.5,2.27,0.0,1.0,24.96,null,null,2022-12-31,Manhattan,West Chelsea/Hudson Yards,Manhattan,East Village


In [0]:
# =============================================================================
# MEMBER 2 — Demand Aggregation & Scoring
# Hengrui Li
# Input:  msbabigdata.spark.trend_market_cleaned_trips  (Member 1 output)
# Output: msbabigdata.spark.trend_market_demand_scores
# =============================================================================
# NOTE ON TIMESTAMPS:
# The cleaned table timestamps have been converted from UTC to NYC time by
# Member 1 (from_utc_timestamp). Because the raw sample data was already in
# NYC local time, the timestamps in the cleaned table are shifted ~5 hours
# earlier than actual pickup times. This does not affect the logic or
# correctness of the pipeline — when the full TLC dataset is loaded, Member 1
# should remove the timezone conversion step, and this notebook requires
# no changes.
# =============================================================================

from pyspark.sql import functions as F
from pyspark.sql import Window

In [0]:
# =============================================================================
# PART 1: Load cleaned trips from Member 1's Delta table
# =============================================================================
 
source_table = "msbabigdata.spark.trend_market_cleaned_trips"
df = spark.table(source_table)
 
# Only select the 4 columns we need — everything else is irrelevant for demand scoring
df = df.select("pulocationid", "pickup_zone", "pickup_borough", "tpep_pickup_datetime")
 
# --- VALIDATION ---
print("=== PART 1: Input Data ===")
print(f"Row count: {df.count()}")
df.show(5)

=== PART 1: Input Data ===
Row count: 2879
+------------+-------------------+--------------+--------------------+
|pulocationid|        pickup_zone|pickup_borough|tpep_pickup_datetime|
+------------+-------------------+--------------+--------------------+
|         142|Lincoln Square East|     Manhattan| 2022-12-31 22:53:13|
|         142|Lincoln Square East|     Manhattan| 2022-12-31 20:30:04|
|         163|      Midtown North|     Manhattan| 2022-12-31 23:56:28|
|         163|      Midtown North|     Manhattan| 2022-12-31 22:03:17|
|         162|       Midtown East|     Manhattan| 2022-12-31 19:35:55|
+------------+-------------------+--------------+--------------------+
only showing top 5 rows


In [0]:
# =============================================================================
# PART 2: Feature Extraction — time_bucket + day_type
# =============================================================================
# TIME BUCKETS (4 segments with business meaning):
#   morning_rush : 06–10  commute to work
#   midday       : 10–16  off-peak daytime
#   evening_rush : 16–20  commute home + dining
#   night        : 20–06  late night / overnight
#
# DAY TYPE (2 categories):
#   weekday  : Monday–Friday  (dayofweek 2–6 in Spark, where 1=Sunday)
#   weekend  : Saturday–Sunday (dayofweek 1 or 7)
 
def assign_time_bucket(hour_col):
    return (
        F.when((hour_col >= 6)  & (hour_col < 10), "morning_rush")
         .when((hour_col >= 10) & (hour_col < 16), "midday")
         .when((hour_col >= 16) & (hour_col < 20), "evening_rush")
         .otherwise("night")
    )
 
def assign_day_type(dow_col):
    # Spark dayofweek: 1=Sunday, 2=Monday, ..., 7=Saturday
    return F.when(dow_col.isin(1, 7), "weekend").otherwise("weekday")
 
df_features = (
    df
    .withColumn("hour", F.hour("tpep_pickup_datetime"))
    .withColumn("dow",  F.dayofweek("tpep_pickup_datetime"))
    .withColumn("time_bucket", assign_time_bucket(F.col("hour")))
    .withColumn("day_type",    assign_day_type(F.col("dow")))
)
 
# --- VALIDATION ---
print("\n=== PART 2: Feature Extraction ===")
print("Time bucket distribution:")
df_features.groupBy("time_bucket").count().orderBy("count", ascending=False).show()
print("Day type distribution:")
df_features.groupBy("day_type").count().orderBy("count", ascending=False).show()


=== PART 2: Feature Extraction ===
Time bucket distribution:
+------------+-----+
| time_bucket|count|
+------------+-----+
|       night| 1189|
|      midday|  717|
|morning_rush|  490|
|evening_rush|  483|
+------------+-----+

Day type distribution:
+--------+-----+
|day_type|count|
+--------+-----+
| weekday| 2070|
| weekend|  809|
+--------+-----+



In [0]:
# =============================================================================
# PART 3: Zone Aggregation — trip_count per (zone × time_bucket × day_type)
# =============================================================================
 
df_agg = (
    df_features
    .groupBy("pulocationid", "pickup_zone", "pickup_borough", "time_bucket", "day_type")
    .agg(
        F.count("*").alias("trip_count"),
        F.countDistinct(F.to_date("tpep_pickup_datetime")).alias("days_observed")
    )
)
 
# --- VALIDATION ---
print("\n=== PART 3: Zone Aggregation ===")
print(f"Aggregated rows (zone × time_bucket × day_type cells): {df_agg.count()}")
print("Sample — top zones by trip count:")
df_agg.orderBy("trip_count", ascending=False).show(10)
 


=== PART 3: Zone Aggregation ===
Aggregated rows (zone × time_bucket × day_type cells): 437
Sample — top zones by trip count:
+------------+--------------------+--------------+------------+--------+----------+-------------+
|pulocationid|         pickup_zone|pickup_borough| time_bucket|day_type|trip_count|days_observed|
+------------+--------------------+--------------+------------+--------+----------+-------------+
|         132|         JFK Airport|        Queens|       night| weekday|        55|           38|
|         186|Penn Station/Madi...|     Manhattan|       night| weekday|        48|           34|
|          48|        Clinton East|     Manhattan|       night| weekday|        47|           34|
|         236|Upper East Side N...|     Manhattan|       night| weekday|        38|           32|
|          79|        East Village|     Manhattan|       night| weekday|        37|           29|
|         132|         JFK Airport|        Queens|       night| weekend|        36|      

In [0]:
# =============================================================================
# PART 4: Demand Score Computation
# =============================================================================
# demand_score = trip_count / avg(trip_count across all zones for same time window)
#
# score > 1.0  →  above average demand for this time window
# score = 1.5  →  50% above average
# score < 1.0  →  below average
#
# We use a Window partitioned by (time_bucket, day_type) so the comparison is
# always apples-to-apples: a zone is only compared against other zones in the
# same time window, not against the global average.
 
window_spec = Window.partitionBy("time_bucket", "day_type")
 
df_scored = (
    df_agg
    .withColumn("avg_trip_count", F.avg("trip_count").over(window_spec))
    .withColumn("stddev_trip_count", F.stddev("trip_count").over(window_spec))
    .withColumn(
        "demand_score",
        F.round(F.col("trip_count") / F.col("avg_trip_count"), 4)
    )
)
 
# --- VALIDATION ---
print("\n=== PART 4: Demand Scores ===")
print("Score distribution summary:")
df_scored.select("demand_score").summary().show()
print("Top 10 highest-scoring zones:")
df_scored.orderBy("demand_score", ascending=False).show(10)


=== PART 4: Demand Scores ===
Score distribution summary:
+-------+------------------+
|summary|      demand_score|
+-------+------------------+
|  count|               437|
|   mean|0.9999967963386754|
| stddev|0.9371938060164426|
|    min|            0.0928|
|    25%|            0.3006|
|    50%|            0.7101|
|    75%|            1.4545|
|    max|            6.3905|
+-------+------------------+

Top 10 highest-scoring zones:
+------------+--------------------+--------------+------------+--------+----------+-------------+------------------+------------------+------------+
|pulocationid|         pickup_zone|pickup_borough| time_bucket|day_type|trip_count|days_observed|    avg_trip_count| stddev_trip_count|demand_score|
+------------+--------------------+--------------+------------+--------+----------+-------------+------------------+------------------+------------+
|         132|         JFK Airport|        Queens|       night| weekend|        36|           18| 5.633333333333334

In [0]:
 
# =============================================================================
# PART 5: Low-Confidence Flagging
# =============================================================================
# A zone-time cell is flagged as low_confidence if ANY of:
#   (a) trip_count < 3      — too few trips, pattern not reliable
#   (b) days_observed < 2   — only seen on 1 day, could be an anomaly
#   (c) CoV > 2.0           — coefficient of variation too high (stddev / mean),
#                             demand is too erratic to trust
#                             CoV = stddev / avg across zones in same window
#
# NOTE ON THRESHOLDS:
# These thresholds are intentionally low because the sample data is small.
# With full TLC data (millions of rows), raise trip_count threshold to ~20
# and days_observed to ~10 for more conservative filtering.
 
TRIP_COUNT_MIN  = 3
DAYS_OBSERVED_MIN = 2
COV_MAX = 2.0
 
df_final = (
    df_scored
    .withColumn(
        "coeff_of_variation",
        F.round(
            F.when(
                F.col("avg_trip_count") > 0,
                F.col("stddev_trip_count") / F.col("avg_trip_count")
            ).otherwise(None),
            4
        )
    )
    .withColumn(
        "is_low_confidence",
        (F.col("trip_count") < TRIP_COUNT_MIN) |
        (F.col("days_observed") < DAYS_OBSERVED_MIN) |
        (F.col("coeff_of_variation") > COV_MAX)
    )
    # Clean up intermediate columns not needed downstream
    .drop("stddev_trip_count")
)
 
# --- VALIDATION ---
print("\n=== PART 5: Low-Confidence Flagging ===")
flag_counts = df_final.groupBy("is_low_confidence").count()
flag_counts.show()
 
total = df_final.count()
low_conf = df_final.filter(F.col("is_low_confidence") == True).count()
print(f"Low-confidence cells: {low_conf} / {total} ({low_conf/total*100:.1f}%)")
print(f"High-confidence cells available for ranking: {total - low_conf}")
 
print("\nSample of flagged zones:")
df_final.filter(F.col("is_low_confidence") == True) \
        .select("pickup_zone", "time_bucket", "day_type", "trip_count", "days_observed", "demand_score", "is_low_confidence") \
        .show(5)
 
print("\nSample of high-confidence zones:")
df_final.filter(F.col("is_low_confidence") == False) \
        .orderBy("demand_score", ascending=False) \
        .select("pickup_zone", "pickup_borough", "time_bucket", "day_type", "trip_count", "demand_score", "is_low_confidence") \
        .show(10)


=== PART 5: Low-Confidence Flagging ===
+-----------------+-----+
|is_low_confidence|count|
+-----------------+-----+
|            false|  268|
|             true|  169|
+-----------------+-----+

Low-confidence cells: 169 / 437 (38.7%)
High-confidence cells available for ranking: 268

Sample of flagged zones:
+-------------------+------------+--------+----------+-------------+------------+-----------------+
|        pickup_zone| time_bucket|day_type|trip_count|days_observed|demand_score|is_low_confidence|
+-------------------+------------+--------+----------+-------------+------------+-----------------+
|     Yorkville East|evening_rush| weekday|         2|            2|      0.3006|             true|
|      South Jamaica|evening_rush| weekday|         1|            1|      0.1503|             true|
|       Central Park|evening_rush| weekday|         1|            1|      0.1503|             true|
|  Battery Park City|evening_rush| weekday|         1|            1|      0.1503|      

In [0]:
# =============================================================================
# PART 6: Write to Delta Lake
# =============================================================================
# Output schema for Member 3 (Heuristic Ranking):
#   pulocationid        — zone ID (join key)
#   pickup_zone         — human-readable zone name
#   pickup_borough      — borough name
#   time_bucket         — morning_rush / midday / evening_rush / night
#   day_type            — weekday / weekend
#   trip_count          — raw pickup count in this cell
#   days_observed       — number of distinct days this cell was observed
#   avg_trip_count      — citywide average for this time window
#   demand_score        — trip_count / avg_trip_count (>1.0 = above average)
#   coeff_of_variation  — stddev/mean across zones in window (signal reliability)
#   is_low_confidence   — boolean flag for Member 3 to filter out
 
target_table = "msbabigdata.spark.trend_market_demand_scores"
 
(
    df_final
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(target_table)
)
 
# --- VALIDATION ---
print("\n=== PART 6: Delta Lake Write Validation ===")
df_check = spark.table(target_table)
print(f"Rows written to {target_table}: {df_check.count()}")
print("\nFinal table schema:")
df_check.printSchema()
print("\nFinal table sample (top demand zones, high confidence only):")
display(
    spark.sql(f"""
        SELECT pickup_zone, pickup_borough, time_bucket, day_type,
               trip_count, demand_score, is_low_confidence
        FROM {target_table}
        WHERE is_low_confidence = false
        ORDER BY demand_score DESC
        LIMIT 15
    """)
)
 


=== PART 6: Delta Lake Write Validation ===
Rows written to msbabigdata.spark.trend_market_demand_scores: 437

Final table schema:
root
 |-- pulocationid: long (nullable = true)
 |-- pickup_zone: string (nullable = true)
 |-- pickup_borough: string (nullable = true)
 |-- time_bucket: string (nullable = true)
 |-- day_type: string (nullable = true)
 |-- trip_count: long (nullable = true)
 |-- days_observed: long (nullable = true)
 |-- avg_trip_count: double (nullable = true)
 |-- demand_score: double (nullable = true)
 |-- coeff_of_variation: double (nullable = true)
 |-- is_low_confidence: boolean (nullable = true)


Final table sample (top demand zones, high confidence only):


pickup_zone,pickup_borough,time_bucket,day_type,trip_count,demand_score,is_low_confidence
JFK Airport,Queens,night,weekend,36,6.3905,false
Upper East Side South,Manhattan,morning_rush,weekday,34,5.1864,false
JFK Airport,Queens,night,weekday,55,5.1058,false
JFK Airport,Queens,evening_rush,weekday,33,4.9595,false
Penn Station/Madison Sq West,Manhattan,night,weekday,48,4.4559,false
Clinton East,Manhattan,night,weekday,47,4.3631,false
Midtown Center,Manhattan,midday,weekday,35,3.8439,false
Midtown Center,Manhattan,morning_rush,weekday,24,3.661,false
JFK Airport,Queens,evening_rush,weekend,11,3.5328,false
Upper East Side North,Manhattan,night,weekday,38,3.5276,false


# Heuristic ranking layer starts here>>

In [0]:
##Heuristic ranking

from pyspark.sql import functions as F
from pyspark.sql import Window

CLEANED_TRIPS_TABLE = "msbabigdata.spark.trend_market_cleaned_trips"
DEMAND_SCORES_TABLE = "msbabigdata.spark.trend_market_demand_scores"
RANKED_ZONES_TABLE  = "msbabigdata.spark.trend_market_ranked_zones"
VALIDATION_TABLE    = "msbabigdata.spark.trend_market_validation_scores"

TOP_N = 5

print(" Configuration loaded successfully")

 Configuration loaded successfully


In [0]:
df_trips = spark.table(CLEANED_TRIPS_TABLE).select(
    "pulocationid", "pickup_zone", "pickup_borough",
    "tpep_pickup_datetime", "pickup_date"
)

# Jan/Feb for ranking, March for validation
df_train = df_trips.filter(F.month("tpep_pickup_datetime").isin(1, 2))
df_val   = df_trips.filter(F.month("tpep_pickup_datetime") == 3)

print(f"Jan/Feb rows (for ranking)  : {df_train.count():,}")
print(f"March rows (for validation) : {df_val.count():,}")

Jan/Feb rows (for ranking)  : 1,916
March rows (for validation) : 951


In [0]:
# aggregate_demand function — uses assign_time_bucket and assign_day_type
def aggregate_demand(df):
    df_feat = (
        df
        .withColumn("hour", F.hour("tpep_pickup_datetime"))
        .withColumn("dow",  F.dayofweek("tpep_pickup_datetime"))
        .withColumn("time_bucket", assign_time_bucket(F.col("hour")))
        .withColumn("day_type",    assign_day_type(F.col("dow")))
    )
    df_agg = (
        df_feat
        .groupBy("pulocationid", "pickup_zone", "pickup_borough",
                 "time_bucket", "day_type")
        .agg(
            F.count("*").alias("trip_count"),
            F.countDistinct(F.to_date("tpep_pickup_datetime")).alias("days_observed")
        )
    )
    win = Window.partitionBy("time_bucket", "day_type")
    return (
        df_agg
        .withColumn("avg_trip_count",    F.avg("trip_count").over(win))
        .withColumn("stddev_trip_count", F.stddev("trip_count").over(win))
        .withColumn("demand_score",
                    F.round(F.col("trip_count") / F.col("avg_trip_count"), 4))
        .withColumn("coeff_of_variation",
                    F.round(
                        F.when(F.col("avg_trip_count") > 0,
                               F.col("stddev_trip_count") / F.col("avg_trip_count")
                        ).otherwise(None), 4
                    ))
        .withColumn("is_low_confidence",
                    (F.col("trip_count") < 3) |
                    (F.col("days_observed") < 2) |
                    (F.col("coeff_of_variation") > 2.0))
        .drop("stddev_trip_count")
    )

print("Helper functions defined successfully")

Helper functions defined successfully


In [0]:
# Score Jan/Feb training data
df_train_scores = aggregate_demand(df_train)

print(f"Jan/Feb demand score cells   : {df_train_scores.count():,}")
print("\nTop zones by demand score (Jan/Feb):")
df_train_scores.filter(F.col("is_low_confidence") == False) \
               .orderBy("demand_score", ascending=False) \
               .show(10, truncate=False)

Jan/Feb demand score cells   : 392

Top zones by demand score (Jan/Feb):
+------------+----------------------------+--------------+------------+--------+----------+-------------+-----------------+------------+------------------+-----------------+
|pulocationid|pickup_zone                 |pickup_borough|time_bucket |day_type|trip_count|days_observed|avg_trip_count   |demand_score|coeff_of_variation|is_low_confidence|
+------------+----------------------------+--------------+------------+--------+----------+-------------+-----------------+------------+------------------+-----------------+
|237         |Upper East Side South       |Manhattan     |morning_rush|weekday |24        |16           |4.647058823529412|5.1646      |0.9398            |false            |
|132         |JFK Airport                 |Queens        |night       |weekend |20        |12           |4.018867924528302|4.9765      |0.879             |false            |
|132         |JFK Airport                 |Queens        

In [0]:
# Save March validation scores
df_val_scores = aggregate_demand(df_val)
(
    df_val_scores
    .write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(VALIDATION_TABLE)
)
print(f"Validation scores (March) written to: {VALIDATION_TABLE}")
print(f"Validation rows: {df_val_scores.count():,}")

Validation scores (March) written to: msbabigdata.spark.trend_market_validation_scores
Validation rows: 310


In [0]:
# Filter low-confidence zones
df_confident = df_train_scores.filter(F.col("is_low_confidence") == False)

total   = df_train_scores.count()
kept    = df_confident.count()
dropped = total - kept

print(f"Total cells     : {total:,}")
print(f"Low-confidence  : {dropped:,}  ({dropped/total*100:.1f}%)")
print(f"High-confidence : {kept:,}  — these proceed to ranking")

Total cells     : 392
Low-confidence  : 173  (44.1%)
High-confidence : 219  — these proceed to ranking


In [0]:
# Rank top 5 zones per time window
rank_window = Window.partitionBy("time_bucket", "day_type").orderBy(
    F.col("demand_score").desc(),
    F.col("trip_count").desc()
)

df_ranked = (
    df_confident
    .withColumn("rank", F.row_number().over(rank_window))
    .filter(F.col("rank") <= TOP_N)
    .select(
        "time_bucket", "day_type", "rank", "pulocationid",
        "pickup_zone", "pickup_borough", "trip_count",
        "avg_trip_count", "demand_score", "days_observed"
    )
    .orderBy("day_type", "time_bucket", "rank")
)

print(f"Ranked shortlist rows (top {TOP_N} per window): {df_ranked.count()}")
df_ranked.show(50, truncate=False)

Ranked shortlist rows (top 5 per window): 40
+------------+--------+----+------------+----------------------------+--------------+----------+------------------+------------+-------------+
|time_bucket |day_type|rank|pulocationid|pickup_zone                 |pickup_borough|trip_count|avg_trip_count    |demand_score|days_observed|
+------------+--------+----+------------+----------------------------+--------------+----------+------------------+------------+-------------+
|evening_rush|weekday |1   |132         |JFK Airport                 |Queens        |22        |4.791666666666667 |4.5913      |18           |
|evening_rush|weekday |2   |79          |East Village                |Manhattan     |15        |4.791666666666667 |3.1304      |12           |
|evening_rush|weekday |3   |138         |LaGuardia Airport           |Queens        |11        |4.791666666666667 |2.2957      |8            |
|evening_rush|weekday |4   |237         |Upper East Side South       |Manhattan     |10        |4

In [0]:
# Save ranked zones
(
    df_ranked
    .write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(RANKED_ZONES_TABLE)
)
print(f"Ranked zones written to: {RANKED_ZONES_TABLE}")
print(f"Total rows written: {spark.table(RANKED_ZONES_TABLE).count()}")

Ranked zones written to: msbabigdata.spark.trend_market_ranked_zones
Total rows written: 40


In [0]:
# Spot check — Manhattan weekday evening rush
print("=== SPOT CHECK: Weekday evening rush, Manhattan ===")
spot_check = (
    df_ranked
    .filter(
        (F.col("time_bucket") == "evening_rush") &
        (F.col("day_type")    == "weekday") &
        (F.col("pickup_borough") == "Manhattan")
    )
    .select("rank", "pickup_zone", "pickup_borough", "demand_score", "trip_count")
)
spot_check.show(truncate=False)

manhattan_count = spot_check.count()
if manhattan_count == 0:
    print("WARNING: No Manhattan zones in weekday evening rush — check your data!")
else:
    print(f"PASS: {manhattan_count} Manhattan zone(s) found in weekday evening rush")

=== SPOT CHECK: Weekday evening rush, Manhattan ===
+----+-------------------------+--------------+------------+----------+
|rank|pickup_zone              |pickup_borough|demand_score|trip_count|
+----+-------------------------+--------------+------------+----------+
|2   |East Village             |Manhattan     |3.1304      |15        |
|4   |Upper East Side South    |Manhattan     |2.087       |10        |
|5   |Times Sq/Theatre District|Manhattan     |2.087       |10        |
+----+-------------------------+--------------+------------+----------+

PASS: 3 Manhattan zone(s) found in weekday evening rush


In [0]:
print("=== Full Ranked Shortlist: All Time Buckets x Day Types ===")
(
    df_ranked
    .select("day_type", "time_bucket", "rank", "pickup_zone", "pickup_borough", "demand_score", "trip_count")
    .orderBy("day_type", "time_bucket", "rank")
    .show(100, truncate=False)
)

=== Full Ranked Shortlist: All Time Buckets x Day Types ===
+--------+------------+----+----------------------------+--------------+------------+----------+
|day_type|time_bucket |rank|pickup_zone                 |pickup_borough|demand_score|trip_count|
+--------+------------+----+----------------------------+--------------+------------+----------+
|weekday |evening_rush|1   |JFK Airport                 |Queens        |4.5913      |22        |
|weekday |evening_rush|2   |East Village                |Manhattan     |3.1304      |15        |
|weekday |evening_rush|3   |LaGuardia Airport           |Queens        |2.2957      |11        |
|weekday |evening_rush|4   |Upper East Side South       |Manhattan     |2.087       |10        |
|weekday |evening_rush|5   |Times Sq/Theatre District   |Manhattan     |2.087       |10        |
|weekday |midday      |1   |Midtown Center              |Manhattan     |3.6351      |23        |
|weekday |midday      |2   |Upper East Side South       |Manhattan 